In [ ]:
#@title 🚀 Bấm nút Play (▶) bên trái để bật VieNeu GPU Server
GPU_TIER = "Auto (Tự nhận T4 / L4 / A100)" #@param ["Auto (Tự nhận T4 / L4 / A100)", "T4 (Free/Pro - Batch 16)", "L4 (Colab Pro - Batch 32)", "A100 (Colab Pro+ - Batch 32)"]
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_FOLDER = "STV-VieNeu" #@param {type:"string"}
EXPORT_FORMAT = "WAV (Không nén · Nhanh nhất)" #@param ["WAV (Không nén · Nhanh nhất)", "MP3 (128kbps · Nhẹ 9x)", "M4A (128kbps · Nhẹ 9x)"]

import os, re, sys, time, json, glob, shutil, threading, subprocess, urllib.request
import importlib.metadata
from IPython.display import HTML, clear_output, display

tier_map = {
    "Auto (Tự nhận T4 / L4 / A100)": "auto",
    "T4 (Free/Pro - Batch 16)": "T4",
    "L4 (Colab Pro - Batch 32)": "L4",
    "A100 (Colab Pro+ - Batch 32)": "A100",
}
fmt_map = {
    "WAV (Không nén · Nhanh nhất)": "wav",
    "MP3 (128kbps · Nhẹ 9x)": "mp3",
    "M4A (128kbps · Nhẹ 9x)": "m4a",
}
tier = tier_map.get(GPU_TIER, "auto")
export_fmt = fmt_map.get(EXPORT_FORMAT, "wav")
safe_folder = re.sub(r'[<>:"/\\|?*\x00-\x1f]+', "_", str(DRIVE_FOLDER or "STV-VieNeu").strip().strip("/\\")) or "STV-VieNeu"

drive_ready = False
cache_dir = ""
export_dir = ""
drive_display_dir = ""
active_tmp_files = set()

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        if not os.path.isdir("/content/drive/MyDrive"):
            print("☁️ Đang kết nối Google Drive (hãy bấm Cho phép / Allow nếu có hộp thoại)...")
            drive.mount("/content/drive", force_remount=False)
        if os.path.isdir("/content/drive/MyDrive"):
            base_drive = os.path.join("/content/drive/MyDrive", safe_folder)
            cache_dir = os.path.join(base_drive, "Cache")
            export_dir = os.path.join(base_drive, "Exports")
            os.makedirs(cache_dir, exist_ok=True)
            os.makedirs(export_dir, exist_ok=True)
            for stale_tmp in glob.glob(os.path.join(cache_dir, "*.tmp")):
                try:
                    os.remove(stale_tmp)
                except OSError:
                    pass
            drive_display_dir = f"MyDrive/{safe_folder}/Exports"
            drive_ready = True
            print(f"✅ Đã kết nối Google Drive: {drive_display_dir}")
    except Exception as drv_err:
        print(f"⚠️ Không kết nối được Google Drive ({drv_err}), chạy chế độ tạm thời không lưu Drive.")

def atomic_tar_pack(parent_dir, target_name, dest_tar):
    tmp_tar = dest_tar + ".tmp"
    active_tmp_files.add(tmp_tar)
    try:
        subprocess.run(["tar", "-cf", tmp_tar, "-C", parent_dir, target_name], check=True)
        os.replace(tmp_tar, dest_tar)
        return True
    except Exception as exc:
        print(f"⚠️ Bỏ qua lưu cache {os.path.basename(dest_tar)}: {exc}")
        return False
    finally:
        active_tmp_files.discard(tmp_tar)
        if os.path.exists(tmp_tar):
            try:
                os.remove(tmp_tar)
            except OSError:
                pass

print("⏳ [1/4] Đang kiểm tra GPU & công cụ nền (espeak-ng, cloudflared)...")
apt_pkgs = []
if not shutil.which("espeak-ng"):
    apt_pkgs.append("espeak-ng")
if export_fmt in ("mp3", "m4a") and not shutil.which("ffmpeg"):
    apt_pkgs.append("ffmpeg")
if apt_pkgs:
    subprocess.run("apt-get update -qq && apt-get install -y -qq " + " ".join(apt_pkgs) + " > /dev/null 2>&1", shell=True, check=True)

cf_bin = "/usr/local/bin/cloudflared"
cf_cache = os.path.join(cache_dir, "cloudflared") if drive_ready else ""
if not shutil.which("cloudflared"):
    if cf_cache and os.path.isfile(cf_cache):
        shutil.copyfile(cf_cache, cf_bin)
        os.chmod(cf_bin, 0o755)
    else:
        subprocess.run(
            "curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
            shell=True,
            check=True,
        )
        if cf_cache:
            tmp_cf = cf_cache + ".tmp"
            active_tmp_files.add(tmp_cf)
            try:
                shutil.copyfile(cf_bin, tmp_cf)
                os.replace(tmp_cf, cf_cache)
            except Exception:
                pass
            finally:
                active_tmp_files.discard(tmp_cf)
                if os.path.exists(tmp_cf):
                    try:
                        os.remove(tmp_cf)
                    except OSError:
                        pass

repo_dir = "/content/vieneu-tts"
repo_tar = os.path.join(cache_dir, "vieneu_repo.tar") if drive_ready else ""
print("⏳ [2/4] Đang nạp / cài đặt VieNeu-TTS...")
if not os.path.isdir(repo_dir):
    if repo_tar and os.path.isfile(repo_tar):
        try:
            subprocess.run(["tar", "-xf", repo_tar, "-C", "/content"], check=True)
        except Exception:
            shutil.rmtree(repo_dir, ignore_errors=True)
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/pnnbao97/VieNeu-TTS.git", repo_dir], check=True)
        if repo_tar:
            atomic_tar_pack("/content", "vieneu-tts", repo_tar)

try:
    import torch as _torch_probe
    _tv = str(getattr(_torch_probe, "__version__", "")).split("+")[0].split(".")
    _torch_tag = f"torch{_tv[0]}_{_tv[1]}" if len(_tv) >= 2 else "notorch"
except Exception:
    _torch_tag = "notorch"
env_tag = f"py{sys.version_info.major}{sys.version_info.minor}_{_torch_tag}"
wheels_dir = "/content/vieneu_wheels"
wheels_tar = os.path.join(cache_dir, f"wheels_{env_tag}.tar") if drive_ready else ""

restored_wheels = False
if wheels_tar and os.path.isfile(wheels_tar):
    try:
        print("⚡ Đang cài thư viện từ cache Google Drive...")
        shutil.rmtree(wheels_dir, ignore_errors=True)
        subprocess.run(["tar", "-xf", wheels_tar, "-C", "/content"], check=True)
        whl_files = sorted(glob.glob(os.path.join(wheels_dir, "*.whl")))
        if whl_files:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--no-index", "--find-links", wheels_dir, *whl_files], check=True)
            ed_res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--no-build-isolation", "-e", repo_dir])
            if ed_res.returncode != 0:
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", repo_dir], check=True)
            subprocess.run([sys.executable, "-c", "import fastapi, uvicorn, huggingface_hub, tokenizers, vieneu"], check=True)
            restored_wheels = True
    except Exception as whl_err:
        print(f"⚠️ Cache thư viện chưa tương thích ({whl_err}), chuyển sang cài trực tuyến...")
        restored_wheels = False

if not restored_wheels:
    before_pkgs = {
        d.metadata["Name"].lower(): d.version
        for d in importlib.metadata.distributions()
        if d.metadata and d.metadata["Name"]
    }
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", repo_dir, "fastapi", "uvicorn", "huggingface_hub", "tokenizers"], check=True)
    if drive_ready and wheels_tar:
        try:
            after_pkgs = {
                d.metadata["Name"].lower(): d.version
                for d in importlib.metadata.distributions()
                if d.metadata and d.metadata["Name"]
            }
            delta_specs = [
                f"{name}=={ver}"
                for name, ver in after_pkgs.items()
                if name not in ("vieneu", "vieneu-tts") and before_pkgs.get(name) != ver
            ]
            if delta_specs:
                print(f"💾 Đang đóng gói {len(delta_specs)} gói thư viện lên Google Drive cho lần sau...")
                shutil.rmtree(wheels_dir, ignore_errors=True)
                os.makedirs(wheels_dir, exist_ok=True)
                subprocess.run([sys.executable, "-m", "pip", "wheel", "-q", "--no-deps", "--wheel-dir", wheels_dir, *delta_specs], check=True)
                atomic_tar_pack("/content", "vieneu_wheels", wheels_tar)
        except Exception as cache_whl_err:
            print(f"⚠️ Bỏ qua đóng gói cache thư viện: {cache_whl_err}")

runner_path = os.path.join(repo_dir, "vieneu_colab_runner.py")
try:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/trankhiembaolong/quick-novel-translator/main/colab/vieneu_colab_runner.py",
        runner_path,
    )
except Exception as dl_err:
    print(f"⚠️ Không tải được runner mới ({dl_err}), dùng chế độ fallback mặc định.")
    runner_path = ""

hf_dir = "/content/hf_cache"
hf_tar = os.path.join(cache_dir, "hf_cache.tar") if drive_ready else ""
hf_was_cached = False
if hf_tar and os.path.isfile(hf_tar):
    try:
        print("⚡ Đang nạp cache Model AI từ Google Drive...")
        shutil.rmtree(hf_dir, ignore_errors=True)
        subprocess.run(["tar", "-xf", hf_tar, "-C", "/content"], check=True)
        if os.path.isdir(hf_dir) and any(os.scandir(hf_dir)):
            hf_was_cached = True
    except Exception as hf_err:
        print(f"⚠️ Không đọc được cache Model AI ({hf_err}), sẽ tải mới...")
        shutil.rmtree(hf_dir, ignore_errors=True)
os.makedirs(hf_dir, exist_ok=True)

print("⏳ [3/4] Đang khởi động VieNeu GPU Server & nạp model AI...")
env = os.environ.copy()
env["HOST"] = "127.0.0.1"
env["PORT"] = "8000"
env["VIENEU_COLAB_GPU_TIER"] = tier
env["VIENEU_WATERMARK"] = "0"
env["PYTHONUNBUFFERED"] = "1"
env["HF_HOME"] = hf_dir
env["HF_HUB_DISABLE_TELEMETRY"] = "1"
if drive_ready:
    env["VIENEU_DRIVE_EXPORT_DIR"] = export_dir
    env["VIENEU_DRIVE_EXPORT_DISPLAY"] = drive_display_dir
    env["VIENEU_DRIVE_EXPORT_FORMAT"] = export_fmt

server_log = open("/content/vieneu_server.log", "w", encoding="utf-8")
cmd = [sys.executable, runner_path] if (runner_path and os.path.exists(runner_path)) else [sys.executable, "-m", "apps.openai_speech"]
server_proc = subprocess.Popen(cmd, cwd=repo_dir, env=env, stdout=server_log, stderr=subprocess.STDOUT)

health_data = {}
deadline = time.time() + 600
while time.time() < deadline:
    if server_proc.poll() is not None:
        server_log.flush()
        raise RuntimeError("VieNeu server thoát sớm. Log:\n" + open("/content/vieneu_server.log", encoding="utf-8").read()[-2000:])
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2) as resp:
            if resp.status == 200:
                health_data = json.loads(resp.read().decode("utf-8"))
                break
    except Exception:
        pass
    time.sleep(2)

if not health_data:
    raise TimeoutError("VieNeu server khởi động quá 10 phút.")

if drive_ready and hf_tar and not hf_was_cached and os.path.isdir(hf_dir) and any(os.scandir(hf_dir)):
    def _bg_cache_hf():
        if atomic_tar_pack("/content", "hf_cache", hf_tar):
            print("💾 Đã lưu xong cache Model AI lên Google Drive (từ lần sau khởi động ~30-45s)!", flush=True)
    threading.Thread(target=_bg_cache_hf, daemon=True).start()

print("⏳ [4/4] Đang mở Cloudflare HTTPS Tunnel miễn phí...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

tunnel_url = ""
tunnel_deadline = time.time() + 60
while time.time() < tunnel_deadline:
    line = tunnel_proc.stdout.readline()
    if not line:
        if tunnel_proc.poll() is not None:
            break
        time.sleep(0.2)
        continue
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        tunnel_url = m.group(0)
        break

if not tunnel_url:
    raise RuntimeError("Không lấy được link trycloudflare.com từ cloudflared.")

gpu_label = health_data.get("gpu_name", "Colab GPU")
batch_label = health_data.get("batch_max_size", health_data.get("export_max_streams", 16))
dtype_label = health_data.get("model_dtype", "auto")
runtime_warn_html = ""
if tier in ("L4", "A100") and "T4" in gpu_label.upper():
    runtime_warn_html = f"<p style=\"margin: 8px 0 0; padding: 6px 10px; border-radius: 6px; background: #fef3c7; color: #92400e; font-size: 12px;\">💡 Bạn đang chọn <b>{tier}</b> nhưng Colab đang cấp phần cứng <b>{gpu_label}</b>. Để đổi lên L4/A100, bấm <b>Runtime → Change runtime type</b>.</p>"
drive_badge_html = (
    f"<p style=\"margin: 8px 0 0; padding: 6px 10px; border-radius: 6px; background: #d1fae5; color: #065f46; font-size: 12px;\">☁️ <b>Drive Cache & Lưu trực tiếp:</b> <code>{drive_display_dir}</code> ({export_fmt.upper()})</p>"
    if drive_ready
    else ""
)

clear_output(wait=True)
card_html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; max-width: 620px; border: 2px solid #10b981; border-radius: 14px; padding: 18px 20px; background: linear-gradient(135deg, #ecfdf5 0%, #f0fdf4 100%); color: #064e3b;">
  <div style="display: flex; align-items: center; justify-content: space-between; gap: 10px; flex-wrap: wrap;">
    <div style="font-size: 16px; font-weight: 800;">🟢 VieNeu Colab GPU đã sẵn sàng!</div>
    <span style="background: #059669; color: #fff; padding: 4px 10px; border-radius: 999px; font-size: 12px; font-weight: 700;">{gpu_label} · Batch {batch_label} · {dtype_label}</span>
  </div>
  <p style="margin: 10px 0 12px; font-size: 13.5px; color: #065f46;">Bấm nút bên dưới để copy link, sau đó quay lại Extension Quick Novel Translator và bấm <b>📋 Dán</b> rồi <b>⬇ Tải Audio (WAV)</b>.</p>
  <div style="display: flex; gap: 8px; align-items: center; flex-wrap: wrap;">
    <input id="stv-colab-url" type="text" readonly value="{tunnel_url}" style="flex: 1; min-width: 240px; padding: 10px 12px; border: 1px solid #34d399; border-radius: 8px; font-size: 13px; font-weight: 600; background: #fff; color: #0f172a;" />
    <button id="stv-copy-btn" onclick="(function(){{var inp=document.getElementById('stv-colab-url');var btn=document.getElementById('stv-copy-btn');var done=function(){{btn.innerText='✓ Đã Copy! Quay lại Extension bấm Dán';btn.style.background='#047857';}};if(navigator.clipboard&&navigator.clipboard.writeText){{navigator.clipboard.writeText(inp.value).then(done).catch(function(){{inp.select();document.execCommand('copy');done();}});}}else{{inp.select();document.execCommand('copy');done();}}}})()" style="padding: 11px 18px; border: none; border-radius: 8px; background: #10b981; color: #fff; font-size: 14px; font-weight: 700; cursor: pointer;">📋 Copy Link Kết Nối</button>
  </div>
  {drive_badge_html}
  {runtime_warn_html}
  <p style="margin: 10px 0 0; font-size: 12px; color: #047857;">⚠️ Giữ tab Colab này mở trong lúc Extension đang tạo Audio.</p>
</div>
"""
display(HTML(card_html))

log_pos = os.path.getsize("/content/vieneu_server.log") if os.path.exists("/content/vieneu_server.log") else 0
try:
    while server_proc.poll() is None and tunnel_proc.poll() is None:
        if os.path.exists("/content/vieneu_server.log"):
            with open("/content/vieneu_server.log", "r", encoding="utf-8", errors="ignore") as lf:
                lf.seek(log_pos)
                chunk = lf.read()
                log_pos = lf.tell()
            for log_line in chunk.splitlines():
                if "BATCH done:" in log_line or "BATCH sent:" in log_line or "💾 DRIVE saved:" in log_line or "ERROR" in log_line:
                    print(log_line, flush=True)
        time.sleep(2)
except KeyboardInterrupt:
    pass
finally:
    for p in (tunnel_proc, server_proc):
        if p.poll() is None:
            p.terminate()
    for tmp_f in list(active_tmp_files):
        if os.path.exists(tmp_f):
            try:
                os.remove(tmp_f)
            except OSError:
                pass
